In [ ]:
import os
import gc
import gzip
import shutil
import subprocess as sp
from multiprocessing import Pool

import numpy as np
import pandas as pd
import nibabel as nib
import tifffile
import ants
from scipy.ndimage import zoom
from numba import njit
import matplotlib.pyplot as plt

In [ ]:
class mapping_to_atlas():
    def __init__(self, data_atlas_path, ants_dir_name, data_ants_path, before_ants_file):
        self.ants_voxel_unit = Resize_um
        self.original_voxel_unit = Resize_um
        self.img_voxel_unit = Resize_um

        self.atlas_tif_path = data_atlas_path
        self.atlas_nii_path = self.atlas_tif_path.replace(".tif", ".nii.gz")

        # Output / working directories and paths
        self.ants_dst_dir = os.path.join(data_ants_path, ants_dir_name)
        self.sample_resize_img_path = os.path.join(data_ants_path, before_ants_file)

        # From .tif to .nii path mapping for the sample
        self.sample_nii_path = os.path.join(
            data_ants_path, before_ants_file.replace(".tif", ".nii.gz")
        )

        self.before_ants_np_path = os.path.join(data_ants_path, "all_points_um.npy")

        if not os.path.exists(self.ants_dst_dir):
            print(f"make ants folder {self.ants_dst_dir}")
            os.makedirs(self.ants_dst_dir)
        else:
            print(f"{self.ants_dst_dir} already exists")

        self.moving_nii_path = self.sample_nii_path
        self.output_nii_path = os.path.join(self.ants_dst_dir, "after_ants.nii.gz")

        # ANTs binary prefix
        self.prefix_ants = "/opt/ANTs/bin/"

    def tif2nii(self, tif_path, nii_path, nii_voxel_unit):
        """Convert a TIFF stack to NIfTI and set voxel units in the header."""
        tif_img = tifffile.imread(tif_path)
        # If axes already match (Z, Y, X), no swap; preserve original behavior
        nii_img = nib.Nifti1Image(tif_img, affine=None)
        aff = np.diag([-nii_voxel_unit, -nii_voxel_unit, nii_voxel_unit, 1])
        nii_img.header.set_qform(aff, code=2)
        nii_img.to_filename(nii_path)
        return

    def run_antsRegistration(self, prefix_ants, atlas_file, moving_file, dst_dir, threads):
        """Run antsRegistration (affine + SyN)."""
        cmd = "ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS={THREADS} && "
        cmd += "{EXECUTABLE} -d 3 "
        cmd += "--initial-moving-transform [{ATLAS_FILE},{MOVING_FILE},1] "
        cmd += "--interpolation Linear "
        cmd += "--use-histogram-matching 0 "
        cmd += "--winsorize-image-intensities [0.05,1.0] "
        cmd += "--float 0 "
        cmd += "--output [{DST_PREFIX},{WARPED_FILE},{INVWARPED_FILE}] "
        cmd += (
            "--transform Affine[0.1] --metric MI[{ATLAS_FILE},{MOVING_FILE},1,128,Regular,0.5] "
            "--convergence [10000x10000x10000,1e-5,15] --shrink-factors 4x2x1 --smoothing-sigmas 2x1x0vox "
        )
        cmd += (
            "--transform SyN[0.1,3.0,0.0] --metric CC[{ATLAS_FILE},{MOVING_FILE},1,5] "
            "--convergence [300x100x30,1e-6,10] --shrink-factors 4x2x1 --smoothing-sigmas 2x1x0vox"
        )

        cmd = cmd.format(
            THREADS=threads,
            EXECUTABLE=os.path.join(prefix_ants, "antsRegistration"),
            DST_PREFIX=os.path.join(dst_dir, "F2M_"),
            WARPED_FILE=os.path.join(dst_dir, "F2M_Warped.nii.gz"),
            INVWARPED_FILE=os.path.join(dst_dir, "F2M_InvWarped.nii.gz"),
            ATLAS_FILE=atlas_file,
            MOVING_FILE=moving_file,
        )
        print(f"[*] Executing : {cmd}")
        sp.call(cmd, shell=True)
        return

    def run_antsApplyTransformsToPoints(self, prefix_ants, src_csv, dst_csv, ANTs_image_dir):
        """Apply inverse transforms to a CSV of points."""
        cmd = "{EXECUTABLE} "
        cmd += "-d 3 "
        cmd += "-i {SRC_CSV} "
        cmd += "-o {DST_CSV} "
        cmd += "-t [{AFFINE_MAT},1] "
        cmd += "-t {INVWARP_NII}"
        cmd = cmd.format(
            EXECUTABLE=os.path.join(prefix_ants, "antsApplyTransformsToPoints"),
            AFFINE_MAT=os.path.join(ANTs_image_dir, "F2M_0GenericAffine.mat"),
            INVWARP_NII=os.path.join(ANTs_image_dir, "F2M_1InverseWarp.nii.gz"),
            SRC_CSV=src_csv,
            DST_CSV=dst_csv,
        )
        # Suppress stdout
        with open(os.devnull, "w", encoding="utf-8") as devnull:
            sp.check_call(cmd, shell=True, stdout=devnull)
        return

    def run_antsApplyTransforms(self, prefix_ants, src_file, atlas_file, dst_file, ANTs_image_dir):
        """Apply transforms to an image (scalar)."""
        cmd = "{EXECUTABLE} "
        cmd += "-d 3 "
        cmd += "-e 0 "  # 0/1/2/3 => scalar/vector/tensor/time-series
        cmd += "-i {SRC_FILE} "
        cmd += "-r {REF} "
        cmd += "-o {DST_FILE} "
        cmd += "-n {INTERP} "
        cmd += "-t {INVWARP_NII} "
        cmd += "-t {AFFINE_MAT} "
        cmd = cmd.format(
            EXECUTABLE=os.path.join(prefix_ants, "antsApplyTransforms"),
            SRC_FILE=src_file,
            REF=atlas_file,
            DST_FILE=dst_file,
            INTERP="Linear",
            INVWARP_NII=os.path.join(ANTs_image_dir, "F2M_1Warp.nii.gz"),
            AFFINE_MAT=os.path.join(ANTs_image_dir, "F2M_0GenericAffine.mat"),
        )
        # Suppress stdout
        with open(os.devnull, "w", encoding="utf-8") as devnull:
            sp.check_call(cmd, shell=True, stdout=devnull)
        return


# 2D slice TIFF images -> 3D stack TIFF
def stack_2d_images(path):
    imgs = os.listdir(path)
    img0 = tifffile.imread(os.path.join(path, imgs[0]))

    width = img0.shape[0]
    height = img0.shape[1]

    stack_image = np.zeros((width, height, len(imgs)))
    for i, img_f in enumerate(imgs):
        if "ANTs" in img_f or "ANTsR" in img_f:
            continue
        img = tifffile.imread(os.path.join(path, img_f))
        stack_image[:, :, i] = img
    return stack_image


def convert_nii_to_tiff(nii_file, output_tiff_file):
    """Load a NIfTI file and save as a TIFF stack."""
    nii_data = nib.load(nii_file)
    nii_array = nii_data.get_fdata()
    tifffile.imwrite(output_tiff_file, nii_array)


def decompress_gzip(input_file, output_file):
    """Decompress a .gz file to a regular file."""
    with gzip.open(input_file, "rb") as f_in:
        with open(output_file, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

In [ ]:
# make higher-resolved images (20 µm)
def multi_save_re(cores, savedir, exp, sample, block, cr_num, fol):
    args = []
    for i in range(cr_num + 1):
        args.append((i, savedir, exp, sample, block, fol))
    with Pool(processes=cores) as pool:
        pool.map(save_points, args)


def multi_save(cores, savedir, exp, sample, block, cr_num, fol):
    args = []
    for i in range(cr_num):
        args.append((i, savedir, exp, sample, block, fol))
    with Pool(processes=cores) as pool:
        pool.map(save_points, args)


def save_points(args):
    i, savedir, exp, sample, block, fol = args

    dst_npy = os.path.join(savedir, exp, sample, fol, f"points_pre_all_{i}")
    np.save(dst_npy, points[i * block : (i + 1) * block])

    print("save:", dst_npy, points[i * block : (i + 1) * block])


def multi_points_re(cores, savedir, exp, sample, moved_points_dir, cr_num, fol):
    args = []
    for i in range(cr_num + 1):
        args.append((i, savedir, exp, sample, moved_points_dir, fol))
    with Pool(processes=cores) as pool:
        pool.map(ants_points, args)


def multi_points(cores, savedir, exp, sample, moved_points_dir, cr_num, fol):
    args = []
    for i in range(cr_num):
        args.append((i, savedir, exp, sample, moved_points_dir, fol))
    with Pool(processes=cores) as pool:
        pool.map(ants_points, args)


def ants_points(args):
    i, savedir, exp, sample, moved_points_dir, fol = args

    moved_points_csv = f"cell_table_after_all_{i}.csv"
    intense_points_pkl = f"cell_table_intense_all_{i}.pkl"

    intense_pkl_path = os.path.join(moved_points_dir, intense_points_pkl)
    moved_csv_path = os.path.join(moved_points_dir, moved_points_csv)
    src_npy_path = os.path.join(savedir, exp, sample, fol, f"points_pre_all_{i}.npy")
    moving_points_path = os.path.join(savedir, exp, sample, fol, f"points_all_{i}.csv")

    if not os.path.exists(intense_pkl_path):
        print("save i:", i)

        points_all_pre = np.load(src_npy_path).astype("uint64")

        # Build DataFrame with axis order X<Z
        points_all = pd.DataFrame(
            {
                "X(um)": points_all_pre[:, 2],
                "Y(um)": points_all_pre[:, 1],
                "Z(um)": points_all_pre[:, 0],
            }
        ).astype("uint16")

        points_all.to_csv(
            moving_points_path,
            index=False,
            header=True,
            chunksize=50000,
            columns=["X(um)", "Y(um)", "Z(um)"],
        )
        del points_all
        gc.collect()

        # Apply transforms (atlas <- moving)
        Mapping.run_antsApplyTransformsToPoints(
            Mapping.prefix_ants, moving_points_path, moved_csv_path, Mapping.ants_dst_dir
        )

        # Load moved points (skip the 1st header row from ANTs tool)
        df_moved = pd.read_csv(
            moved_csv_path,
            skiprows=1,
            header=None,
            names=["X(um)", "Y(um)", "Z(um)"],
            dtype={"X(um)": np.uint16, "Y(um)": np.uint16, "Z(um)": np.uint16},
        )

        # Swap X and Z (X < Z)
        temp = df_moved["X(um)"].copy()
        df_moved["X(um)"] = df_moved["Z(um)"]
        df_moved["Z(um)"] = temp
        del temp
        gc.collect()

        # Attach intensity column
        df_moved["intensity"] = points_all_pre[:, 3]

        # Save as pickle (X>Z)
        df_moved.to_pickle(intense_pkl_path)

    else:
        print("skip: ", i)


@njit
def construct_voxel_grid(data, voxel_grid):
    """Fill a voxel grid with max intensity per voxel."""
    for i in range(data.shape[0]):
        x, y, z = int(data[i, 0]), int(data[i, 1]), int(data[i, 2])  # explicit cast to int
        value = data[i, 3]
        voxel_grid[z, y, x] = max(voxel_grid[z, y, x], value)
    return


@njit
def fill_zero_voxels_with_average(voxel_grid):
    """Fill zero-valued voxels with the mean of non-zero neighbors within a 3×3×3 window."""
    print("filling zero voxels")
    depth, height, width = voxel_grid.shape
    output_grid = voxel_grid.copy()

    for z in range(depth):
        for y in range(height):
            for x in range(width):
                if voxel_grid[z, y, x] == 0:
                    total_intensity = 0.0
                    count = 0
                    # 3x3x3 neighborhood
                    for dz in range(-1, 2):
                        for dy in range(-1, 2):
                            for dx in range(-1, 2):
                                nz, ny, nx = z + dz, y + dy, x + dx
                                if (
                                    0 <= nz < depth
                                    and 0 <= ny < height
                                    and 0 <= nx < width
                                    and (dz, dy, dx) != (0, 0, 0)
                                ):
                                    val = voxel_grid[nz, ny, nx]
                                    if val != 0:
                                        total_intensity += val
                                        count += 1
                    if count > 0:
                        output_grid[z, y, x] = total_intensity / count

    return output_grid

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"
cfos_dir = os.path.join(src, "circadian_1st", "circadian_1st_Reconst")
savedir = os.path.join(dst, "cfos_app")

In [ ]:
# Collect unique sample names and classify SYTOX-G / cfos paths

CT_li = np.arange(0, 48, 4)  # circadian time points (CT0–44, every 4 h)
sample_ids = np.arange(1, 7, 1)

# Initialize containers
reconsts = os.listdir(cfos_dir)
sample_names = []
data_parent_paths = []
data_moving_paths = []

for CT in CT_li:
    if CT != 0:
        continue
    for sample_id in sample_ids:
        if sample_id > 1:
            continue
        sample = f"CT{CT}_{str(sample_id).zfill(2)}"

        for reconst in reconsts:
            if sample in reconst:
                sample_names.append(sample)
                for color in os.listdir(os.path.join(cfos_dir, reconst)):
                    color_path = os.path.join(cfos_dir, reconst, color)
                    if "cfos" in color:
                        data_moving_paths.append(color_path)
                    else:
                        data_parent_paths.append(color_path)

print(len(sample_names))
print(len(data_parent_paths))
print(len(data_moving_paths))

In [ ]:
# Prepare stacks, normalize/resize, and run ANTs registration

exp = "1st"
organ_name = "Brain"
Resize_um = 50
threads = 20
rx = 1 / 4

data_atlas_path = os.path.join(src, "CUBIC_R_atlas_ver5", f"iso_{Resize_um}um_R.tif")
before_ants_file = f"before_ants_{Resize_um}um_R.tif"
before_ants_file_raw = "raw_stacked.tif"
before_ants_file_raw_norm = "raw_stacked_norm.tif"
ants_dir_name = "ANTsR50"

for i, data_parent_path in enumerate(data_parent_paths):
    # Target sample name
    sample = sample_names[i]
    print(sample)

    # --- SYTOX-G side (reference image) ---
    sample_dir = os.path.join(savedir, exp, sample)
    sytox_dir = os.path.join(sample_dir, "SYTOX-G")
    os.makedirs(sytox_dir, exist_ok=True)

    sytox_before_path = os.path.join(sytox_dir, before_ants_file)
    sytox_raw_path = os.path.join(sytox_dir, before_ants_file_raw)
    sytox_raw_norm_path = os.path.join(sytox_dir, before_ants_file_raw_norm)

    if not os.path.exists(sytox_before_path):
        if not os.path.exists(sytox_raw_path):
            stacked_image = stack_2d_images(data_parent_path)
            # Align axes (keep original behavior)
            stacked_image = stacked_image.swapaxes(1, 0)
            stacked_image = stacked_image.swapaxes(2, 0)
            tifffile.imwrite(sytox_raw_path, stacked_image)
        else:
            stacked_image = tifffile.imread(sytox_raw_path)

        # Normalize by median of high-intensity voxels (mask values < cutoff)
        cutoff = 5000
        norm_value = 10000
        flip_image2 = stacked_image.copy()
        flip_image2[flip_image2 < cutoff] = np.nan
        median = np.nanmedian(flip_image2)

        stacked_image = (stacked_image / median) * norm_value
        tifffile.imwrite(sytox_raw_norm_path, stacked_image)

        # Resize (zoom)
        stacked_image = zoom(stacked_image, (rx, rx, rx), order=3)

        tifffile.imwrite(sytox_before_path, stacked_image)
        del flip_image2
        del stacked_image
        gc.collect()

    # --- cfos side (moving image) ---
    data_moving_path = data_moving_paths[i]
    cfos_dir = os.path.join(sample_dir, "cfos")
    os.makedirs(cfos_dir, exist_ok=True)

    cfos_raw_path = os.path.join(cfos_dir, before_ants_file_raw)
    cfos_raw_norm_path = os.path.join(cfos_dir, before_ants_file_raw_norm)
    cfos_before_path = os.path.join(cfos_dir, before_ants_file)

    # Always rebuild stack as in original (commented conditions were disabled)
    stacked_image = stack_2d_images(data_moving_path)
    stacked_image = stacked_image.swapaxes(1, 0)
    stacked_image = stacked_image.swapaxes(2, 0)
    tifffile.imwrite(cfos_raw_path, stacked_image)

    cutoff = 5000
    norm_value = 10000
    flip_image2 = stacked_image.copy()
    flip_image2[flip_image2 < cutoff] = np.nan
    median = np.nanmedian(flip_image2)

    stacked_image = (stacked_image / median) * norm_value
    tifffile.imwrite(cfos_raw_norm_path, stacked_image)

    stacked_image = zoom(stacked_image, (rx, rx, rx), order=3)
    tifffile.imwrite(cfos_before_path, stacked_image)

    del flip_image2
    del stacked_image
    gc.collect()

    # --- ANTs registration / application ---
    data_ants_path_p = sytox_dir  # reference image directory
    Mapping = mapping_to_atlas(data_atlas_path, ants_dir_name, data_ants_path_p, before_ants_file)

    if not os.path.exists(Mapping.sample_nii_path):
        Mapping.tif2nii(Mapping.sample_resize_img_path, Mapping.sample_nii_path, Mapping.ants_voxel_unit)
    if not os.path.exists(Mapping.atlas_nii_path):
        Mapping.tif2nii(Mapping.atlas_tif_path, Mapping.atlas_nii_path, Mapping.ants_voxel_unit)

    # Run registration if transform not yet produced
    affine_mat_path = os.path.join(Mapping.ants_dst_dir, "F2M_0GenericAffine.mat")
    if not os.path.exists(affine_mat_path):
        Mapping.run_antsRegistration(
            Mapping.prefix_ants, Mapping.atlas_nii_path, Mapping.sample_nii_path, Mapping.ants_dst_dir, threads
        )
        print("ANTs registration end")

    # Apply transform to cfos image
    data_moving_file = cfos_before_path
    data_moving_nii_path = os.path.join(cfos_dir, before_ants_file.replace(".tif", ".nii.gz"))
    output_nii_path = os.path.join(cfos_dir, ants_dir_name, "after_ants.nii.gz")
    os.makedirs(os.path.join(cfos_dir, ants_dir_name), exist_ok=True)

    Mapping.tif2nii(data_moving_file, data_moving_nii_path, Mapping.ants_voxel_unit)
    Mapping.run_antsApplyTransforms(
        Mapping.prefix_ants, data_moving_nii_path, Mapping.atlas_nii_path, output_nii_path, Mapping.ants_dst_dir
    )
    print("ANTs apply transform end")

In [ ]:
# Check ANTs outputs for cfos

after_ants_file = "after_ants.tif"

for i, sample in enumerate(sample_names):
    if i > 0:
        continue

    sample_dir = os.path.join(savedir, exp, sample, "cfos")
    ants_dir = os.path.join(sample_dir, ants_dir_name)

    gz_file = os.path.join(ants_dir, "after_ants.nii.gz")
    if not os.path.exists(gz_file):
        print(f"{gz_file} not found")
        continue

    nii_file = os.path.join(ants_dir, "after_ants.nii")
    output_tiff_file = os.path.join(ants_dir, after_ants_file)

    # Always regenerate (keeps original behavior)
    decompress_gzip(gz_file, nii_file)
    convert_nii_to_tiff(nii_file, output_tiff_file)

    img = tifffile.imread(output_tiff_file)

    x_num = img.shape[2]
    y_num = img.shape[1]
    z_num = img.shape[0]

    plt.imshow(img[:, :, 150])
    plt.show()

In [ ]:
# Parameter settings for voxel and ANTs configuration

moving_points_csv_pre = "cell_table_all.csv"
moving_points_csv = "cell_table_before_all.csv"
moved_points_csv = "cell_table_after_all.csv"
dir = "img_points"

# Resize / voxel parameters
Resized_um = 50
vx = 10
vx2 = 20  # target voxel size for output dimensions

data_atlas_path = os.path.join(src, "kinoshita/CUBIC_R_atlas_ver5", f"iso_{Resize_um}um_R.tif")
atlas_img = tifffile.imread(data_atlas_path)
x_num = atlas_img.shape[2]
y_num = atlas_img.shape[1]
z_num = atlas_img.shape[0]

ants_dir_name = "ANTsR50"
ants_dir_name_after = "ANTsR20"
before_ants_file = f"before_ants_{Resized_um}um_R.tif"
before_ants_file_raw_norm = "raw_stacked_norm.tif"

# Output volume size after resampling
scale = Resized_um / vx2
x_after = int(x_num * scale)
y_after = int(y_num * scale)
z_after = int(z_num * scale)

cr_num = 28
cores = 14

In [ ]:
# Make and save higher-resolution ANTs-normalized images

for i, sample in enumerate(sample_names):
    after_dir = os.path.join(savedir, exp, sample, "cfos", ants_dir_name_after)
    after_tif = os.path.join(after_dir, "after_ants.tif")
    if os.path.exists(after_tif):
        continue

    print(sample)
    work_dir = os.path.join(savedir, exp, sample, dir)
    os.makedirs(work_dir, exist_ok=True)

    cfos_norm_path = os.path.join(savedir, exp, sample, "cfos", before_ants_file_raw_norm)
    if os.path.exists(cfos_norm_path):
        img = tifffile.imread(cfos_norm_path)

        ind = np.where(np.ravel(np.swapaxes(img > 400, 0, 2)))[0]
        x_ori, y_ori, z_ori = img.shape[2], img.shape[1], img.shape[0]

        intense = np.ravel(img.swapaxes(0, 2))
        points = np.zeros((x_ori * y_ori * z_ori, 3), dtype="uint16")

        c = 0
        for x in range(x_ori):
            for y in range(y_ori):
                for z in range(z_ori):
                    points[c][0] = x * vx
                    points[c][1] = y * vx
                    points[c][2] = z * vx
                    c += 1

        points = points[ind].astype("uint16")
        intense = intense[ind].astype("uint16")
        points = np.concatenate([points, intense.reshape(-1, 1)], axis=1).astype("uint16")
        print(points.shape)

        block = len(points) // cr_num
        block_re = len(points) % cr_num

        data_ants_path_p = os.path.join(savedir, exp, sample, "SYTOX-G")
        Mapping = mapping_to_atlas(data_atlas_path, ants_dir_name, data_ants_path_p, before_ants_file)
        moved_points_dir = os.path.join(Mapping.ants_dst_dir, dir)
        os.makedirs(moved_points_dir, exist_ok=True)

        if __name__ == "__main__":
            if block_re != 0:
                multi_save_re(cores, savedir, exp, sample, block, cr_num, dir)
            else:
                multi_save(cores, savedir, exp, sample, block, cr_num, dir)

            if block_re != 0:
                multi_points_re(cores, savedir, exp, sample, moved_points_dir, cr_num, dir)
            else:
                multi_points(cores, savedir, exp, sample, moved_points_dir, cr_num, dir)

        img_out = np.zeros((z_after, y_after, x_after), dtype=np.uint16)

        if block_re != 0:
            rng = range(cr_num + 1)
        else:
            rng = range(cr_num)

        for j in rng:
            intense_points_pkl = f"cell_table_intense_all_{j}.pkl"
            df = pd.read_pickle(os.path.join(moved_points_dir, intense_points_pkl))
            df["X(um)"] = (df["X(um)"] / vx2).astype("int64").astype("uint16")
            df["Y(um)"] = (df["Y(um)"] / vx2).astype("int64").astype("uint16")
            df["Z(um)"] = (df["Z(um)"] / vx2).astype("int64").astype("uint16")
            df["intensity"] = df["intensity"].fillna(0).astype(np.uint16)
            df = df[
                (df["X(um)"].between(0, x_after - 1))
                & (df["Y(um)"].between(0, y_after - 1))
                & (df["Z(um)"].between(0, z_after - 1))
            ]
            data_array = df[["X(um)", "Y(um)", "Z(um)", "intensity"]].to_numpy()
            construct_voxel_grid(data_array, img_out)

        img_out = fill_zero_voxels_with_average(img_out)
        os.makedirs(after_dir, exist_ok=True)
        tifffile.imwrite(after_tif, img_out)
        print("save:", after_tif)
        del img_out
        gc.collect()